In [1]:
import os
import urllib.request
from urllib.parse import quote
import gradio as gr
from dotenv import load_dotenv

# LangChain 관련 컴포넌트 임포트
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
api_key = os.getenv("GOOGLE_API_KEY")

# 1. 예제 PDF 파일 다운로드
url = quote("https://raw.githubusercontent.com/ai-agent-kr/agent-tutorial/main/Ch02/2020_경제금융용어 700선_게시.pdf", safe=":/")
filename = "2020_경제금융용어 700선_게시.pdf"
urllib.request.urlretrieve(url, filename=filename)

c:\agent-prg\ch02\ch02_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\HKIT\AppData\Local\Temp\ipykernel_15296\125021270.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


('2020_경제금융용어 700선_게시.pdf', <http.client.HTTPMessage at 0x2c96d460e10>)

In [2]:
loader = PyPDFLoader("2020_경제금융용어 700선_게시.pdf")
texts = loader.load_and_split()

print(f"로드된 초기 청크(페이지) 수: {len(texts)}")
# 실행 결과: 366

로드된 초기 청크(페이지) 수: 366


In [3]:
texts[15]

Document(metadata={'producer': 'Smart PDF Imposition 1.0', 'creator': 'Smart PDF Imposition 1.0', 'creationdate': 'January 31, 2018 16:21', 'author': 'INSUNG DATA INC.', 'creationdate--text': 'January 31, 2018 16:21', 'moddate': 'January 31, 2018 16:21', 'moddate--text': 'January 31, 2018 16:21', 'subject': 'CoreImposition PDF 1.0 Output', 'title': '°æÁ¦±ÝÀ¶¿ë¾î 700¼±-ÃÖÁ¾.PDF', 'source': '2020_경제금융용어 700선_게시.pdf', 'total_pages': 371, 'page': 19, 'page_label': '20'}, page_content='3\nㄱ \n신용을 모두 포괄하는 것으로 크게 가계대출과 판매신용으로 구분된다. 현재 가계신용통\n계는 2002년 말 잔액부터 분기별로 제공되고 있다. 가계신용통계는 우리나라 가계부문의 \n부채 규모 및 변동 등을 파악하는 데 널리 활용되고 있다 .\n가계처분가능소득 \n가계처분가능소득(PDI; Personal Disposable Income)은 가계가 맘대로 소비와 저축\n으로 처분할 수 있는 소득을 의미한다. 흔히 국민들의 생활수준을 파악해 볼 수 있는 \n지표로 1인당 GNI가 널리 쓰이고 있으나 국민총소득에는 가계 뿐 아니라 기업 금융기관 \n정부가 벌어 들인 소득이 모두 포함되어 있다. 따라서 기업과 금융기관 등이 가계부문 \n보다 더 많은 소득을 벌어 1인당 국민총소득(GNI)이 높아진 경우에는 가계가 느끼는 \n체감경기는 전체 경기와 괴리가 있게 된다. 1인당 가계총처분가능소득(PGDI; Personal \nGross Disposable Income)은 가계부문의 총처분가능소득을 연앙인구로 나누

In [4]:
# 15번 청크의 메타데이터 및 본문 일부 확인
print(texts[15].metadata)
# {'source': '2020_경제금융용어 700선_게시.pdf', 'page': 19}
print(texts[15].page_content[:150])
print(texts[0].page_content)

{'producer': 'Smart PDF Imposition 1.0', 'creator': 'Smart PDF Imposition 1.0', 'creationdate': 'January 31, 2018 16:21', 'author': 'INSUNG DATA INC.', 'creationdate--text': 'January 31, 2018 16:21', 'moddate': 'January 31, 2018 16:21', 'moddate--text': 'January 31, 2018 16:21', 'subject': 'CoreImposition PDF 1.0 Output', 'title': '°æÁ¦±ÝÀ¶¿ë¾î 700¼±-ÃÖÁ¾.PDF', 'source': '2020_경제금융용어 700선_게시.pdf', 'total_pages': 371, 'page': 19, 'page_label': '20'}
3
ㄱ 
신용을 모두 포괄하는 것으로 크게 가계대출과 판매신용으로 구분된다. 현재 가계신용통
계는 2002년 말 잔액부터 분기별로 제공되고 있다. 가계신용통계는 우리나라 가계부문의 
부채 규모 및 변동 등을 파악하는 데 널리 활용되고 있다 .
가계처분가능소득 
가계처분
iii
찾아보기
한국은행은 국민들이 경제 및 금융에 대한 이해도를 높이고 경제에 관한 합리적인 
의사결정 능력을 키울 수 있도록 현장 경제교육, 온라인 경제교육, 경제교육 콘텐츠 
개발 등 대국민 경제교육을 다양하게 수행해 오고 있습니다 . 
이의 일환으로 2018년 
 경제금융용어 700선
 을 발간하였는데 그간 동 책자에 대한 
수요가 꾸준히 늘어남에 따라 이번에 추가로 발간하게 되었습니다 .
지난번 내용과 같이 통화정책, 실물경제, 금융안정, 지급결제 등 한국은행 주요 
업무를 이해하는데 필요한 전문 용어와 경제·금융 흐름을 이해하는데 도움이 되는 
시사 경제금융 용어들을 수록하였습니다. 용어해설은 개념과 도입 배경, 의미, 적용 
사례 등을 담아 쉽게 이해할 수 있도록 하였습니다. 또한 e-b

In [5]:
print(texts[13].page_content)

1
ㄱ 
ㄱ
가계부실위험지수(HDRI)
가구의 소득 흐름은 물론 금융 및 실물 자산까지 종합적으로 고려하여 가계부채의 
부실위험을 평가하는 지표로, 가계의 채무상환능력을 소득 측면에서 평가하는 원리금상
환비율(DSR; Debt Service Ratio)과 자산 측면에서 평가하는 부채/자산비율(DTA; Debt 
To Asset Ratio)을 결합하여 산출한 지수이다. 가계부실위험지수는 가구의 DSR과 DTA가 
각각 40%, 100%일 때 100의 값을 갖도록 설정되어 있으며, 동 지수가 100을 초과하는 
가구를 ‘위험가구’로 분류한다. 위험가구는 소득 및 자산 측면에서 모두 취약한 ‘고위험가구’, 
자산 측면에서 취약한 ‘고DTA가구’, 소득 측면에서 취약한 ‘고DSR가구’로 구분할 수 있다. 
다만 위험 및 고위험 가구는 가구의 채무상환능력 취약성 정도를 평가하기 위한 것이며 
이들 가구가 당장 채무상환 불이행, 즉 임계상황에 직면한 것을 의미하지 않는다 .
 연관검색어 : 총부채원리금상환비율(DSR)
가계수지
가정에서 일정 기간의 수입(명목소득)과 지출을 비교해서 남았는지 모자랐는지를 
표시한 것을 가계수지(household's total income and expenditure)라 한다. 가계수지가 
가계수지 ∙


In [6]:
texts = texts[13:]
print("줄어든 문서갯수:",len(texts))

줄어든 문서갯수: 353


In [7]:
print(texts[0])

page_content='1
ㄱ 
ㄱ
가계부실위험지수(HDRI)
가구의 소득 흐름은 물론 금융 및 실물 자산까지 종합적으로 고려하여 가계부채의 
부실위험을 평가하는 지표로, 가계의 채무상환능력을 소득 측면에서 평가하는 원리금상
환비율(DSR; Debt Service Ratio)과 자산 측면에서 평가하는 부채/자산비율(DTA; Debt 
To Asset Ratio)을 결합하여 산출한 지수이다. 가계부실위험지수는 가구의 DSR과 DTA가 
각각 40%, 100%일 때 100의 값을 갖도록 설정되어 있으며, 동 지수가 100을 초과하는 
가구를 ‘위험가구’로 분류한다. 위험가구는 소득 및 자산 측면에서 모두 취약한 ‘고위험가구’, 
자산 측면에서 취약한 ‘고DTA가구’, 소득 측면에서 취약한 ‘고DSR가구’로 구분할 수 있다. 
다만 위험 및 고위험 가구는 가구의 채무상환능력 취약성 정도를 평가하기 위한 것이며 
이들 가구가 당장 채무상환 불이행, 즉 임계상황에 직면한 것을 의미하지 않는다 .
 연관검색어 : 총부채원리금상환비율(DSR)
가계수지
가정에서 일정 기간의 수입(명목소득)과 지출을 비교해서 남았는지 모자랐는지를 
표시한 것을 가계수지(household's total income and expenditure)라 한다. 가계수지가 
가계수지 ∙' metadata={'producer': 'Smart PDF Imposition 1.0', 'creator': 'Smart PDF Imposition 1.0', 'creationdate': 'January 31, 2018 16:21', 'author': 'INSUNG DATA INC.', 'creationdate--text': 'January 31, 2018 16:21', 'moddate': 'January 31, 2018 16:21', 'moddate--text': 'January 31, 2018 16:21', 'subject': 'CoreImposition PDF 1.0 Output', 'title': '°æÁ¦±

In [8]:
print(texts[-1])

page_content='경제금융용어 700선
발행인 이주열
편집인 박철원
발행처 한국은행(www.bok.or.kr)
서울특별시 중구 세종대로 67(태평로 2가)
발행일 2020년 8월 3일
인  쇄 ㈜제일프린테크 
 본 자료는 한국은행 홈페이지(http://www.bok.or.kr>경제교육>온라인학습>일반인) 에서 
다운로드 받으실 수 있습니다.
 이 책자에 대한 문의는 한국은행 경제교육실 경제교육기획팀(02-759-5618)으로 연락 
하여 주시기 바랍니다.
 
 경제금융용어 700선
 은 
① 정부간행물판매센터(☎ 02-734-6818, http://www.gpcbooks.co.kr) 
② 한국경제서적(☎ 02-737-7498)
③ 경제서적(☎ 02-736-0640, http://kj-book.co.kr) 또는 주요 서점 등에서 매권당 
8,000원에 판매하고 있습니다.
ISBN 979-11-5538-393-3 03320' metadata={'producer': 'Smart PDF Imposition 1.0', 'creator': 'Smart PDF Imposition 1.0', 'creationdate': 'January 31, 2018 16:21', 'author': 'INSUNG DATA INC.', 'creationdate--text': 'January 31, 2018 16:21', 'moddate': 'January 31, 2018 16:21', 'moddate--text': 'January 31, 2018 16:21', 'subject': 'CoreImposition PDF 1.0 Output', 'title': '°æÁ¦±ÝÀ¶¿ë¾î 700¼±-ÃÖÁ¾.PDF', 'source': '2020_경제금융용어 700선_게시.pdf', 'total_pages': 371, 'page': 369, 'page_label': '370'}


In [9]:
texts = texts[:-1]
print("마지막장 제거 문서갯수:",len(texts))

마지막장 제거 문서갯수: 352


In [10]:
embedding = OpenAIEmbeddings()
db_path = "./chroma_db"

if os.path.exists(db_path) and os.listdir(db_path):
    # 이미 DB가 저장되어 있다면 새로 임베딩하지 않고 기존 DB 로드
    vectordb = Chroma(
        persist_directory=db_path,
        embedding_function=embedding
    )
    print("기존 VectorDB를 로드했습니다.")
else:
    # DB가 없을 때만 최초 1회 생성 및 로컬 저장
    vectordb = Chroma.from_documents(
        documents=texts,
        embedding=embedding,
        persist_directory=db_path
    )
    print("VectorDB를 최초 생성 및 저장했습니다.")

print(f"적재 완료된 청크 수: {vectordb._collection.count()}")
# embedding = OpenAIEmbeddings()
# #embedding = OpenAIEmbeddings(model="text-embedding-3-small")
# print(embedding.model)

# # 벡터 데이터베이스 생성 및 청크 적재
# vectordb = Chroma.from_documents(
#     documents=texts,
#     embedding=embedding
# )

# # 적재된 총 청크(Document) 개수 확인
# print(f"적재 완료된 청크 수: {vectordb._collection.count()}")
# # 실행 결과: 352

C:\Users\HKIT\AppData\Local\Temp\ipykernel_15296\2557021277.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


기존 VectorDB를 로드했습니다.
적재 완료된 청크 수: 352


In [11]:
# 1. 컬렉션 딕셔너리의 키 확인
for key in vectordb._collection.get():
    print(key)
# 반환 키: ['ids', 'embeddings', 'documents', 'uris', 'data', 'metadatas']

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


ids
embeddings
metadatas
documents
uris
data
included


In [12]:
# 임베딩 벡터를 포함하여 조회
data = vectordb._collection.get(include=['embeddings'])
print(f"로드된 벡터 개수: {len(data['embeddings'])}")
print(f"첫 번째 문서의 임베딩 값 출력: {data['embeddings'][0]}")
print(f"첫 번째 문서의 임베딩 차원 수: {len(data['embeddings'][0])}")

로드된 벡터 개수: 352
첫 번째 문서의 임베딩 값 출력: [-0.012334919534623623, -0.015185566619038582, 0.028385741636157036, -0.033402882516384125, -0.0070159463211894035, 0.004493962042033672, -0.01738559640944004, -0.010651360265910625, -0.0192904993891716, -0.05682514235377312, -0.020819786936044693, 0.04231031611561775, -0.01171113085001707, -0.011818449012935162, 0.004889698699116707, 0.04268592968583107, 0.01819048449397087, -0.006928750313818455, 0.0018562745535746217, -0.018445365130901337, -0.005952822510153055, 0.010336112231016159, -0.02128930576145649, -0.003207816742360592, 0.00028443592600524426, 0.009101949632167816, 0.022442979738116264, -0.017358766868710518, 0.006519598420709372, 0.006821431685239077, -0.001489882473833859, -0.012194063514471054, -0.03477119281888008, 0.008317182771861553, -0.006271424237638712, -0.0027886040043085814, 0.007700101938098669, 0.006013189442455769, 0.02401251345872879, -0.015588010661303997, 0.010865997523069382, 0.0010924687376245856, 0.012777607887983322, -

In [13]:
metadatas = vectordb._collection.get()['metadatas']
print('metadatas의 갯수',len(metadatas))
print('첫번째문서의 출처', metadatas[0])

metadatas의 갯수 352
첫번째문서의 출처 {'author': 'INSUNG DATA INC.', 'creationdate': 'January 31, 2018 16:21', 'creationdate--text': 'January 31, 2018 16:21', 'creator': 'Smart PDF Imposition 1.0', 'moddate': 'January 31, 2018 16:21', 'moddate--text': 'January 31, 2018 16:21', 'page': 156, 'page_label': '157', 'producer': 'Smart PDF Imposition 1.0', 'source': '2020_경제금융용어 700선_게시.pdf', 'subject': 'CoreImposition PDF 1.0 Output', 'title': '°æÁ¦±ÝÀ¶¿ë¾î 700¼±-ÃÖÁ¾.PDF', 'total_pages': 371}


In [14]:
# 상위 2개의 연관 청크를 반환하는 검색기 선언
retriever = vectordb.as_retriever(search_kwargs={"k": 2})

In [15]:
query = "비트코인이 궁금해"
docs = retriever.invoke(query)

print(f"반환된 유사 문서 개수: {len(docs)}")
print("유사문서1", docs[0])
print("유사문서2", docs[1])
# for i, doc in enumerate(docs):
#     print(f"[{i+1}] Page: {doc.metadata.get('page')}")
#     print(f"{doc.page_content[:120]}...\n")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


반환된 유사 문서 개수: 2
유사문서1 page_content='139
ㅂ 
비트코인
비트코인(bitcoin)은 가상통화(암호통화)이자 디지털 지급시스템이다. 비트코인 시스템
은 중앙 저장소 또는 단일 관리자가 없기 때문에 최초의 탈중앙화된 디지털통화라고 불린다. 
이는 사토시 나카모토라는 사람(집단)에 의해 만들어져서 2009년 개방형 소프트웨어로 
배포되었다. 이 시스템은 공유형(peer-to-peer)이며, 거래는 중개자 없이 블록체인 소프트
웨어를 이용하는 참여자(nodes) 사이에 직접 이뤄진다. 이런 거래들은 공유(P2P) 네트워크
상 참여자의 작업증명(proof-of-work)을 통해 검증되고 공개된 분산원장인 블록체인에 
기록된다. 승인된 거래의 새 그룹인 한 블록은 대략 10분에 한 번씩 만들어져서 블록체인에 
추가되고 신속하게 모든 참여자에게 보내어진다. 비트코인은 대규모 전기를 사용하는 
컴퓨터 처리 능력(power)을 활용해서 이뤄지는 기록보관 서비스인 채굴(mining)에 대한 
보상으로 발행되고 있으며 다른 통화･상품･용역 등과 교환되어가고 있다. 중앙은행이 
발행한 법정화폐가 아닌 비트코인은 비트코인 플랫폼에서 거래되며 투자대상으로서도 
관심을 받고 있다. 하지만 급등한 가격 및 심한 변동 폭으로 거품논란이 크다. 또한 익명성으
로 자금세탁 등 불법거래에 악용될 소지가 많아 중국 등 일부 국가에서 비트코인 등 가상통
화의 거래를 규제하고 있다. 일본의 경우 비트코인의 거래이익에 대해 세금을 부과한다. 
비트코인은 추가되는 한 블록당 새로 12.5비트코인을 보상하는데(2016.7월 현재), 21만개
가 채굴될 때(대략 4년)마다 그 보상이 반으로 줄어든다. 비트코인의 총량은 21백만개이며 
2140년경 모두 채굴될 것으로 전망된다. 비트코인은 그 시스템의 설계시 그 수량을 한정시
켜 놓았기 때문에 원칙적으로 인플레이션에 의해 가치가 떨어질 수 있는 화폐와 다른 
속성을 지닌다. 한편 2017년 8월 1일 비트코인(classic bitc

In [16]:
template = """당신은 한국은행에서 만든 금융 용어를 설명해 주는 '금융쟁이'입니다.
안상준 개발자가 만들었습니다. 주어진 검색 결과를 바탕으로 답변하세요.
검색 결과에 없는 내용이라면 답변할 수 없다고 하세요. 친절한 어조로 답변하세요.

[검색 결과]
{context}

질문: {question}
답변:"""

prompt = PromptTemplate.from_template(template)

In [17]:
# LLM 모델 초기화 (재현성을 위해 temperature=0 설정)
# 2) LLM만 Gemini로 변경!
# from langchain_google_genai import ChatGoogleGenerativeAI

# llm = ChatGoogleGenerativeAI(
#     model="gemini-3.8-flash",
#     temperature=0.2
# )
llm = ChatOpenAI(model="gpt-4.1", temperature=0)

def query_rag(question: str) -> dict:
    """사용자 질의를 받아 RAG 파이프라인을 실행하는 코어 함수"""
    # 1. 질의와 관련된 문서 검색
    docs = retriever.invoke(question)

    # 2. 검색된 문서 본문 결합
    context = "\n\n".join([doc.page_content for doc in docs])

    # 3. 프롬프트 포맷팅
    formatted_prompt = prompt.format(question=question, context=context)

    # 4. LLM 추론
    response = llm.invoke(formatted_prompt)

    # 5. 질문, 답변, 참조 근거 문서를 함께 반환 (추적성 확보)
    return {
        "query": question,
        "result": response.content,
        "source_documents": docs
    }

In [18]:
input_text = "디커플링이란 무엇인가?"
chatbot_response = query_rag(input_text)
print(chatbot_response)


{'query': '디커플링이란 무엇인가?', 'result': "안녕하세요! 질문해주셔서 감사합니다.\n\n디커플링(decoupling)이란 '탈동조화'라고 번역할 수 있는데요, 어떤 나라나 지역의 경제가 인접한 다른 국가나 전반적인 세계 경제의 흐름과는 다른 모습을 보이는 현상을 말합니다.\n\n예를 들어, 최근 금융위기 이후 신흥국가나 유로지역 국가들이 미국 경제와는 다른 모습을 보인 것이 디커플링의 예라고 할 수 있습니다. 또한, 주가나 금리, 환율 등 일부 경제 변수의 흐름이 국가 간 또는 특정 국가 내에서 서로 다른 흐름을 보이는 현상도 디커플링이라고 합니다. 예를 들어, 미국의 주가와 한국의 주가가 서로 다른 방향으로 움직이거나, 한 국가 내에서 주가는 하락하지만 해당국 통화는 강세를 보이는 경우 등이 이에 해당합니다.\n\n반대로, 한 나라 또는 지역의 경제가 인접한 다른 국가나 세계경제 흐름과 유사한 흐름을 보이는 것을 '커플링(동조화, coupling)'이라고 합니다.\n\n추가로 궁금한 점이 있으시면 언제든 질문해 주세요!", 'source_documents': [Document(metadata={'author': 'INSUNG DATA INC.', 'creationdate': 'January 31, 2018 16:21', 'creationdate--text': 'January 31, 2018 16:21', 'creator': 'Smart PDF Imposition 1.0', 'moddate': 'January 31, 2018 16:21', 'moddate--text': 'January 31, 2018 16:21', 'page': 114, 'page_label': '115', 'producer': 'Smart PDF Imposition 1.0', 'source': '2020_경제금융용어 700선_게시.pdf', 'subject': 'CoreImposition PDF 1.0 Output', 'title': '°æÁ¦±ÝÀ¶¿ë¾î 700¼±-

In [ ]:
def get_chatbot_response(input_text: str) -> str:
    """Gradio 인터페이스 바인딩을 위한 응답 래퍼 함수"""
    chatbot_response = query_rag(input_text)
    return chatbot_response["result"].strip()

# 인터페이스 생성
with gr.Blocks() as demo:
    chatbot = gr.Chatbot(label="경제금융용어 챗봇")
    msg = gr.Textbox(label="질문해주세요!")
    clear = gr.Button("대화 초기화")

    # 챗봇의 답변을 처리하는 함수
    def respond(message, chat_history):
        # 만약 기존 함수가 문자열을 직접 반환한다면: bot_message = get_chatbot_response(message)
        # 슬라이드처럼 딕셔너리({'result': ...})를 반환하는 경우:
        result = get_chatbot_response(message)  # 또는 query_rag(message)
        
        # result가 dict 형태인지 str 형태인지에 따라 처리
        if isinstance(result, dict):
            bot_message = result.get('result', str(result))
        else:
            bot_message = result

        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": bot_message})
        return "", chat_history

    # 입력 제출 시 respond 호출 / 초기화 버튼
    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: None, None, chatbot, queue=False)

# 인터페이스 실행
demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
